# Lost Interpreter: recurrence, numbers and execution

Program string → train-only BPE → embedding128 → packed two-layer BiGRU → masked mean/max → scalar head → SmoothL1 on standardized output → integer prediction.

**Điểm đã đo ở các version Kaggle trước:** R08 val MAE15.3089, public14.1608/private16.7920. Typed instruction R09=15.8136, mixed curriculum R10=16.3178 trên cùng holdout.

**Bài học:** BiGRU nhỏ vượt Transformer trong điều kiện đã thử. Tách opcode/operand chưa đủ học đúng state transitions. Đổi RUN_ID sang9/10 để học typed representation và curriculum; không nhầm với NPI có trace supervision.

Notebook này chứa source đầy đủ nhưng chưa execute tại bản export này; execution_count=null và outputs rỗng là có chủ ý. Không suy diễn metric mới từ notebook chưa chạy.

Bật GPU T4 trên Kaggle, attach competition tương ứng trước Run All. Loader tự tìm cả `/kaggle/input/competitions/<slug>` và `/kaggle/input/<slug>`. Code assert T4, không tự chuyển CPU. Thực hành sau contest, không dùng trong thời gian cuộc thi cấm trợ giúp.


# Lost Interpreter: học biểu diễn phép thực thi

## Bài khó ở đâu?

Input là chuỗi OP1..OP12 cùng số nguyên; target là một số, không có trace của trạng thái trung gian. RNN trên BPE giảm MAE rất mạnh so với Transformer nhỏ. Điều này ủng hộ thử bias tuần tự, chưa chứng minh GRU tự học đúng semantics. R08 sai trung bình 5.91 ở ≤4 lệnh, 16.64 ở 5–8, 22.01 ở 9–16. Pooling token có thể làm mờ lệnh, thứ tự và vai trò operand. Parser numeric cũ còn đọc nhầm chữ số opcode; R03/R07 không phải bằng chứng sạch rằng số học không có ích. R08 không dùng nhánh ấy.

## 5 paper ưu tiên

| Hạng | Paper / nguồn gốc | Ý chính dùng được | Điều kiện không được bỏ qua | Quyết định |
|---|---|---|---|---|
| 1 | [Learning to Execute, Zaremba & Sutskever, 2015](https://arxiv.org/html/1410.4615v3) §3–4, §8 | Recurrent execution; curriculum kết hợp ví dụ dễ với phân phối đầy đủ | Chương trình và supervision khác; paper lưu ý teacher forcing; không đem accuracy digit so MAE | R09 biểu diễn instruction, R10 curriculum hỗn hợp |
| 2 | [Neural Arithmetic Units, Madsen & Johansen, ICLR 2020](https://arxiv.org/pdf/2001.05016) §1–3 | Tách độ lớn số khỏi ký hiệu; phép cộng/nhân cần bias phù hợp, kiểm soát ổn định | NMU không có phép chia; gates khó học; không đủ biết OP là phép gì | R09 đưa operand đúng giá trị. Chưa gọi nhánh này NAU/NMU |
| 3 | [Neural Programmer-Interpreters, Reed & de Freitas, ICLR 2016](https://arxiv.org/html/1511.06279v4) §3 | Tách mã chương trình, arguments, recurrent core | NPI dùng fully supervised execution traces và program memory; bài này chỉ có output cuối | Mượn typed input; không tuyên bố reproduce NPI |
| 4 | [Neural GPUs Learn Algorithms, Kaiser & Sutskever, ICLR 2016](https://arxiv.org/html/1511.08228v3) §2–3 | Repeated shared transitions, kiểm tra khả năng tổng quát theo độ dài | Binary arithmetic và training recipe chuyên biệt; không tự suy ra control flow OP | Chưa chạy Neural GPU nguyên bản; ưu tiên instruction sequence nhỏ hơn |
| 5 | [Learning Simple Algorithms from Examples, Zaremba et al., ICML 2016](https://proceedings.mlr.press/v48/zaremba16.html) | Controller với tape/grid, tách dữ liệu khỏi thao tác | Q-learning + interface khác, cần thiết kế môi trường; quá nhiều biến thay đổi | Để dành nếu lỗi đòi memory/pointer; không thêm RL vào lượt hiện tại |

## Thí nghiệm đã định trước

**R09:** parse `OP11 7 10` thành opcode 11 và operands [7,10], không thành các số 1,7,10. Mỗi instruction: mean embedding BPE64 + opcode32 + từng operand theo vị trí (value/20, signed-log, presence mask) → MLP128 → GRU hai lớp một chiều → masked mean/max + trạng thái cuối → scalar. Không cắt instruction/operand; kích thước padding đo từ cấu trúc input. Train-only BPE2048; SmoothL1 beta .2; holdout/seed giống R08; tối đa 60 epoch/300 giây.

**R10:** giữ nguyên R09, chỉ đổi sampling trong 20 epoch đầu: nửa từ nhóm dễ tăng dần theo số lệnh, nửa từ toàn bộ training. Sau đó toàn bộ phân phối. Không sinh pseudo program/target khi chưa biết interpreter.

R09 đổi granularity và hướng recurrence cùng lúc: nếu tăng không được gán hết cho numeric channel. R10 so với R09 mới là ablation riêng curriculum. Báo MAE theo độ dài, lỗi lớn nhất, thời gian và VRAM. Chưa có kết quả GPU vòng 2 tại thời điểm viết kế hoạch này.


In [ ]:
SOURCE_COMMIT = '84e9a9efaa6346578162fa12915a8368f18e6485'
RUN_ID = 8


## Shared utilities

Seed, BPE, masking, data paths, schema checks và report writer.


In [ ]:
"""Utilities for controlled, idea-driven AICC NLP experiments."""
import ast, csv, hashlib, importlib.metadata, json, math, os, random, time
from copy import deepcopy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

SEED = 42
OUT = Path("/kaggle/working")
DEVICE = torch.device("cuda:0")


def start_run(task, run_id, config):
    assert torch.cuda.is_available(), "GPU T4 is required; do not silently fall back to CPU"
    name = torch.cuda.get_device_name(0)
    assert "T4" in name, "Expected T4, received " + name
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.set_num_threads(2)
    torch.backends.cudnn.benchmark = False
    torch.cuda.reset_peak_memory_stats()
    info = dict(task=task, run_id=f"R{run_id:02d}", config=config, seed=SEED,
                device=name, gpu_count=torch.cuda.device_count(), started=time.time(),
                source_commit=globals().get("SOURCE_COMMIT"),
                versions={p: importlib.metadata.version(p) for p in
                          ["torch", "numpy", "pandas", "scikit-learn", "tokenizers"]})
    print("RUN_START " + json.dumps(info), flush=True)
    return info


def find_data(slug):
    for root in (Path("/kaggle/input/competitions")/slug, Path("/kaggle/input")/slug):
        if root.is_dir():
            return root
    raise FileNotFoundError("Attach competition data: " + slug)


def make_bpe(texts, size=2048):
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    tokenizer.decoder = decoders.BPEDecoder()
    tokenizer.train_from_iterator(texts, trainers.BpeTrainer(
        vocab_size=size, min_frequency=2,
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]))
    return tokenizer


def encode_texts(tokenizer, texts, max_length):
    values = np.zeros((len(texts), max_length), dtype=np.int64)
    for i, encoded in enumerate(tokenizer.encode_batch(texts)):
        ids = encoded.ids[:max_length] or [1]
        values[i, :len(ids)] = ids
    return values


def hash_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def finish_run(info, validation, submission, checks, history, artifacts=None, limitations=None):
    assert checks.get("ids_unique") is True and checks.get("schema_valid") is True
    assert checks.get("row_count", 0) > 0
    submission.to_csv(OUT/"submission.csv", index=False)
    info.update(status="completed", elapsed_seconds=time.time()-info.pop("started"),
                validation=validation, submission_checks=checks, history=history,
                peak_gpu_memory_mb=torch.cuda.max_memory_allocated()/1024**2,
                artifacts=artifacts or {}, limitations=limitations or [],
                leaderboard_public=None, leaderboard_private=None)
    (OUT/"run_report.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
    print("NLP_RUN_RESULT " + json.dumps(info), flush=True)
    print("NLP_RUN_PASSED", flush=True)
    return info


class PositionEncoder(nn.Module):
    def __init__(self, vocab, dim=96, layers=2, max_length=384, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab, dim, padding_idx=0)
        self.position = nn.Embedding(max_length, dim)
        block = nn.TransformerEncoderLayer(dim, 4, dim*3, dropout=dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(block, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, ids):
        mask = ids.eq(0)
        x = self.embedding(ids) + self.position(torch.arange(ids.shape[1], device=ids.device))
        x = self.encoder(x, src_key_padding_mask=mask)
        return self.norm(x), mask


def pools(hidden, mask):
    valid = (~mask).unsqueeze(-1)
    mean = (hidden*valid).sum(1)/valid.sum(1).clamp(min=1)
    maximum = hidden.masked_fill(~valid, -1e4).amax(1)
    return torch.cat([mean, maximum], -1)



## Model và experiments

Các config Rxx nằm đầu source. Đổi RUN_ID trước khi Run All; mỗi lượt phải bắt đầu model/optimizer mới.


In [ ]:
"""Lost Interpreter: random-init Transformer, BPE, structured and numeric ablations."""
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import re


LOST_CONFIGS={
 1:dict(idea="BPE Transformer with mean/max pooling and direct MAE loss",structure=False,numeric=False,attention=False,aux=False,target="standard",layers=2),
 2:dict(idea="Expose instruction boundaries using dedicated separators",structure=True,numeric=False,attention=False,aux=False,target="standard",layers=2),
 3:dict(idea="Numeric summary branch supplements subword representations",structure=True,numeric=True,attention=False,aux=False,target="standard",layers=2),
 4:dict(idea="Learned query attention pooling replaces fixed pooling",structure=True,numeric=False,attention=True,aux=False,target="standard",layers=2),
 5:dict(idea="Auxiliary instruction-count prediction regularizes program structure",structure=True,numeric=False,attention=False,aux=True,target="standard",layers=2),
 6:dict(idea="Signed-log target tests outlier compression under MAE",structure=True,numeric=False,attention=False,aux=False,target="signed_log",layers=2),
 7:dict(idea="Combine structured tokens, numeric branch and query pooling",structure=True,numeric=True,attention=True,aux=True,target="standard",layers=2)
}


LOST_CONFIGS[8]={**LOST_CONFIGS[1],"idea":"Scratch BPE BiGRU with packed sequences, masked pooling and SmoothL1","backbone":"bigru","loss":"smooth_l1"}
LOST_CONFIGS[9]=dict(idea="Typed instruction BPE plus exact operand channels and recurrent execution",curriculum=False,papers=['1410.4615','1511.06279','2001.05016'])
LOST_CONFIGS[10]={**LOST_CONFIGS[9],"idea":"Typed instruction model with mixed easy/full curriculum","curriculum":True}

def program_text(value,structured):
    text=str(value).replace(";"," ; ")
    if structured:
        text=text.replace(";"," [SEP] ")
        text=re.sub(r"(-?\d+)",r" [NUM] \1 ",text)
    return "[CLS] "+text


def numeric_features(program):
    numbers=np.array([float(n) for n in re.findall(r"(?<![A-Za-z0-9_])[-+]?\d+",str(program))],dtype=float)
    if not len(numbers):numbers=np.array([0.])
    return np.array([len(str(program).split(";")),len(numbers),numbers.sum(),numbers.mean(),
                     numbers.min(),numbers.max(),numbers.std(),np.abs(numbers).sum()],dtype=np.float32)


class ProgramTransformer(nn.Module):
    def __init__(self,vocab,config,dim=96):
        super().__init__();self.config=config
        self.encoder=PositionEncoder(vocab,dim,config["layers"],max_length=256)
        if config["attention"]:
            self.query=nn.Parameter(torch.randn(dim)/math.sqrt(dim))
        self.numerical=nn.Sequential(nn.Linear(8,32),nn.GELU(),nn.LayerNorm(32))
        width=dim*2+(32 if config["numeric"] else 0)
        self.head=nn.Sequential(nn.Linear(width,128),nn.GELU(),nn.Dropout(0.1),nn.Linear(128,1))
        self.auxiliary=nn.Linear(dim*2,1)

    def forward(self,ids,numbers):
        h,mask=self.encoder(ids)
        representation=pools(h,mask)
        if self.config["attention"]:
            scores=(h*self.query).sum(-1)/math.sqrt(h.shape[-1])
            weights=scores.masked_fill(mask,-1e4).softmax(-1)
            pooled=(h*weights.unsqueeze(-1)).sum(1)
            representation=torch.cat([pooled,representation[:,h.shape[-1]:]],-1)
        final=torch.cat([representation,self.numerical(numbers)],-1) if self.config["numeric"] else representation
        return self.head(final).squeeze(-1),self.auxiliary(representation).squeeze(-1)


class ProgramBiGRU(nn.Module):
    def __init__(self,vocab,dim=128):
        super().__init__()
        self.embedding=nn.Embedding(vocab,dim,padding_idx=0)
        self.gru=nn.GRU(dim,dim,2,batch_first=True,bidirectional=True,dropout=.1)
        self.head=nn.Sequential(nn.Linear(dim*4,256),nn.GELU(),nn.Dropout(.1),nn.Linear(256,128),nn.GELU(),nn.Linear(128,1))
        self.auxiliary=nn.Linear(dim*4,1)
    def forward(self,ids,numbers):
        lengths=ids.ne(0).sum(1).clamp(min=1).cpu()
        packed=nn.utils.rnn.pack_padded_sequence(self.embedding(ids),lengths,batch_first=True,enforce_sorted=False)
        encoded,_=self.gru(packed)
        h,_=nn.utils.rnn.pad_packed_sequence(encoded,batch_first=True,total_length=ids.shape[1])
        representation=pools(h,ids.eq(0))
        return self.head(representation).squeeze(-1),self.auxiliary(representation).squeeze(-1)


def run_lost(run_id):
    if run_id>=9:return run_lost_typed(run_id)
    config={**LOST_CONFIGS[run_id],"epochs":45,"budget_seconds":300,"batch_size":128,"bpe_vocab":2048,"max_length":192}
    info=start_run("lost-interpreter",run_id,config)
    root=find_data("lost-interpreter-aicc-round-5")
    train=pd.read_csv(root/"train.csv");test=pd.read_csv(root/"test.csv")
    assert {"program","output"}.issubset(train.columns) and {"ID","program"}.issubset(test.columns)
    tr_idx,va_idx=train_test_split(np.arange(len(train)),test_size=0.15,random_state=SEED)
    if run_id>=8:print("TRAIN_PROGRAM_EXAMPLES "+train.iloc[tr_idx[:8]][["program","output"]].to_json(orient="records"),flush=True)
    texts=[program_text(p,config["structure"]) for p in train.program]
    test_texts=[program_text(p,config["structure"]) for p in test.program]
    tokenizer=make_bpe((texts[i] for i in tr_idx),config["bpe_vocab"])
    tokenizer.add_special_tokens(["[NUM]"])
    tokenizer.save(str(OUT/"tokenizer.json"))
    ids=encode_texts(tokenizer,texts,config["max_length"])
    test_ids=encode_texts(tokenizer,test_texts,config["max_length"])
    num=np.stack([numeric_features(p) for p in train.program])
    test_num=np.stack([numeric_features(p) for p in test.program])
    num=np.sign(num)*np.log1p(np.abs(num));test_num=np.sign(test_num)*np.log1p(np.abs(test_num))
    center,scale=num[tr_idx].mean(0),np.maximum(num[tr_idx].std(0),1e-6)
    num=(num-center)/scale;test_num=(test_num-center)/scale
    truth=train.output.to_numpy(dtype=np.float32)
    transformed=np.sign(truth)*np.log1p(np.abs(truth)) if config["target"]=="signed_log" else truth
    ym,ys=float(transformed[tr_idx].mean()),max(float(transformed[tr_idx].std()),1e-6)
    target=(transformed-ym)/ys
    aux=np.array([len(str(p).split(";")) for p in train.program],dtype=np.float32)
    aux=np.log1p(aux)
    am,astd=float(aux[tr_idx].mean()),max(float(aux[tr_idx].std()),1e-6)
    aux=(aux-am)/astd
    tr=TensorDataset(torch.from_numpy(ids[tr_idx]),torch.from_numpy(num[tr_idx]),
                     torch.from_numpy(target[tr_idx]),torch.from_numpy(aux[tr_idx]))
    model=(ProgramBiGRU(tokenizer.get_vocab_size()) if config.get("backbone")=="bigru" else ProgramTransformer(tokenizer.get_vocab_size(),config)).to(DEVICE)
    optim=torch.optim.AdamW(model.parameters(),lr=8e-4,weight_decay=0.01)
    scaler=torch.amp.GradScaler("cuda")
    loader=DataLoader(tr,batch_size=config["batch_size"],shuffle=True,num_workers=0)
    def predict_ids(x,n):
        model.eval();values=[]
        with torch.inference_mode():
            for a,b in DataLoader(TensorDataset(torch.from_numpy(x),torch.from_numpy(n)),batch_size=192):
                with torch.autocast("cuda",dtype=torch.float16):
                    p,_=model(a.to(DEVICE),b.to(DEVICE))
                values.append(p.float().cpu().numpy())
        raw=np.concatenate(values)*ys+ym
        if config["target"]=="signed_log":raw=np.sign(raw)*np.expm1(np.clip(np.abs(raw),0,20))
        return raw
    best=float("inf");best_state=None;history=[];bad=0;tick=time.time()
    for epoch in range(config["epochs"]):
        model.train();losses=[]
        for x,n,y,a in loader:
            optim.zero_grad(set_to_none=True)
            with torch.autocast("cuda",dtype=torch.float16):
                pred,aux_pred=model(x.to(DEVICE),n.to(DEVICE))
                loss=F.smooth_l1_loss(pred,y.to(DEVICE),beta=.2) if config.get("loss")=="smooth_l1" else F.l1_loss(pred,y.to(DEVICE))
                if config["aux"]:loss=loss+0.05*F.mse_loss(aux_pred,a.to(DEVICE))
            assert torch.isfinite(loss)
            scaler.scale(loss).backward();scaler.unscale_(optim)
            nn.utils.clip_grad_norm_(model.parameters(),1);scaler.step(optim);scaler.update()
            losses.append(float(loss.detach()))
        pred=predict_ids(ids[va_idx],num[va_idx])
        mae=float(mean_absolute_error(truth[va_idx],pred))
        row=dict(epoch=epoch+1,train_loss=float(np.mean(losses)),validation_mae=mae)
        history.append(row);print("EPOCH "+json.dumps(row),flush=True)
        if mae<best:
            best=mae;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};bad=0
        else:bad+=1
        if bad>=9 or time.time()-tick>config["budget_seconds"]:break
    model.load_state_dict(best_state)
    val=predict_ids(ids[va_idx],num[va_idx]);pred=predict_ids(test_ids,test_num)
    assert np.isfinite(pred).all()
    validation_cases=pd.DataFrame(dict(row_index=va_idx,program=train.program.iloc[va_idx].tolist(),truth=truth[va_idx],prediction=val))
    validation_cases["absolute_error"]=np.abs(val-truth[va_idx])
    validation_cases["instructions"]=validation_cases.program.map(lambda p:len(str(p).split(";")))
    validation_cases.to_csv(OUT/"validation_predictions.csv",index=False)
    print("VALIDATION_WORST "+validation_cases.nlargest(8,"absolute_error").to_json(orient="records"),flush=True)
    by_length=validation_cases.groupby(pd.cut(validation_cases.instructions,[0,4,8,16,10000]),observed=True).absolute_error.agg(["mean","count"])
    print("ERROR_BY_LENGTH "+by_length.to_json(),flush=True)
    pd.DataFrame(dict(row_index=tr_idx)).to_csv(OUT/"train_indices.csv",index=False)
    torch.save(dict(state=best_state,y_mean=ym,y_std=ys,numeric_mean=center,numeric_std=scale),OUT/"model.pt")
    sub=pd.DataFrame(dict(ID=test.ID,output=np.rint(pred).astype(np.int64)))
    checks=dict(ids_unique=sub.ID.is_unique,schema_valid=list(sub.columns)==["ID","output"],
                row_count=len(sub),ids_match=bool(sub.ID.equals(test.ID)),finite=bool(np.isfinite(pred).all()))
    return finish_run(info,dict(metric="MAE",value=best,direction="minimize",samples=len(va_idx),
          split="fixed_train_holdout_15pct"),sub,checks,history,
          artifacts=dict(tokenizer="tokenizer.json",pretrained_weights=False,
                         train_sha256=hash_file(root/"train.csv"),test_sha256=hash_file(root/"test.csv")),
          limitations=["Single fixed holdout. Results depend on optimization budget.",
                       "The official source uses a different split; its MAE is not a paired comparison."])


def parse_instructions(program):
    parsed=[]
    for instruction in str(program).split(';'):
        parts=instruction.strip().split()
        if not parts:continue
        match=re.fullmatch(r'OP(\d+)',parts[0])
        assert match, 'Unexpected opcode: '+parts[0]
        opcode=int(match.group(1));assert 1<=opcode<=12
        values=[int(x) for x in parts[1:]]
        parsed.append((opcode,values,instruction.strip()))
    assert parsed
    return parsed


class TypedProgramGRU(nn.Module):
    def __init__(self,vocab,operand_width,dim=128):
        super().__init__()
        self.embedding=nn.Embedding(vocab,64,padding_idx=0)
        self.opcode=nn.Embedding(13,32,padding_idx=0)
        self.instruction=nn.Sequential(nn.Linear(96+operand_width,dim),nn.GELU(),nn.LayerNorm(dim))
        self.gru=nn.GRU(dim,dim,2,batch_first=True,dropout=.1)
        self.head=nn.Sequential(nn.Linear(dim*3,256),nn.GELU(),nn.Dropout(.1),nn.Linear(256,1))
    def forward(self,ids,opcodes,operands):
        valid=ids.ne(0).unsqueeze(-1)
        lexical=(self.embedding(ids)*valid).sum(2)/valid.sum(2).clamp(min=1)
        x=self.instruction(torch.cat([lexical,self.opcode(opcodes),operands],-1))
        lengths=opcodes.ne(0).sum(1).clamp(min=1).cpu()
        packed=nn.utils.rnn.pack_padded_sequence(x,lengths,batch_first=True,enforce_sorted=False)
        h,last=self.gru(packed)
        h,_=nn.utils.rnn.pad_packed_sequence(h,batch_first=True,total_length=opcodes.shape[1])
        representation=torch.cat([pools(h,opcodes.eq(0)),last[-1]],-1)
        return self.head(representation).squeeze(-1)


def run_lost_typed(run_id):
    cfg={**LOST_CONFIGS[run_id],"epochs":60,"budget_seconds":300,"batch_size":128,"bpe_vocab":2048,"pretrained":False}
    info=start_run('lost-interpreter',run_id,cfg);root=find_data('lost-interpreter-aicc-round-5')
    train=pd.read_csv(root/'train.csv');test=pd.read_csv(root/'test.csv')
    tr,va=train_test_split(np.arange(len(train)),test_size=.15,random_state=SEED)
    programs=[parse_instructions(p) for p in list(train.program)+list(test.program)]
    assert parse_instructions('OP11 7 10 ; OP12 -3')[0][:2]==(11,[7,10])
    tokenizer=make_bpe((inst[2] for i in tr for inst in programs[i]),cfg['bpe_vocab'])
    tokenizer.save(str(OUT/'tokenizer.json'))
    max_steps=max(map(len,programs));max_args=max(len(x[1]) for p in programs for x in p)
    max_tokens=max(len(tokenizer.encode(x[2]).ids) for p in programs for x in p)
    ids=np.zeros((len(programs),max_steps,max_tokens),dtype=np.int64)
    ops=np.zeros((len(programs),max_steps),dtype=np.int64)
    values=np.zeros((len(programs),max_steps,max_args*3),dtype=np.float32)
    for i,p in enumerate(programs):
        for j,(op,args,text) in enumerate(p):
            token_ids=tokenizer.encode(text).ids;ids[i,j,:len(token_ids)]=token_ids;ops[i,j]=op
            for k,value in enumerate(args):
                values[i,j,k]=value/20
                values[i,j,max_args+k]=np.sign(value)*np.log1p(abs(value))/3
                values[i,j,2*max_args+k]=1
    assert np.isfinite(values).all()
    n=len(train);truth=train.output.to_numpy(np.float32);ym=float(truth[tr].mean());ys=max(float(truth[tr].std()),1e-6)
    target=(truth-ym)/ys
    model=TypedProgramGRU(tokenizer.get_vocab_size(),max_args*3).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=8e-4,weight_decay=.01);scaler=torch.amp.GradScaler('cuda')
    @torch.inference_mode()
    def predict(indices):
        model.eval();out=[]
        for batch in np.array_split(indices,max(1,math.ceil(len(indices)/256))):
            with torch.autocast('cuda',dtype=torch.float16):
                out.append(model(torch.from_numpy(ids[batch]).to(DEVICE),torch.from_numpy(ops[batch]).to(DEVICE),torch.from_numpy(values[batch]).to(DEVICE)).float().cpu().numpy())
        return np.concatenate(out)*ys+ym
    rng=np.random.default_rng(SEED);lengths=np.array(list(map(len,programs)))
    best=float('inf');state=None;history=[];bad=0;tick=time.time()
    for epoch in range(cfg['epochs']):
        model.train();order=rng.permutation(tr)
        if cfg['curriculum'] and epoch<20:
            # Half easy examples and half unrestricted; hard examples are never excluded.
            cap=np.quantile(lengths[tr],min(1.,.25+.75*epoch/20))
            easy=tr[lengths[tr]<=cap]
            order=np.concatenate([rng.choice(easy,len(tr)//2,replace=True),rng.choice(tr,len(tr)-len(tr)//2,replace=True)])
            rng.shuffle(order)
        losses=[]
        for indices in np.array_split(order,math.ceil(len(order)/cfg['batch_size'])):
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):
                p=model(torch.from_numpy(ids[indices]).to(DEVICE),torch.from_numpy(ops[indices]).to(DEVICE),torch.from_numpy(values[indices]).to(DEVICE))
                loss=F.smooth_l1_loss(p,torch.from_numpy(target[indices]).to(DEVICE),beta=.2)
            assert torch.isfinite(loss)
            scaler.scale(loss).backward();scaler.unscale_(optimizer);nn.utils.clip_grad_norm_(model.parameters(),1)
            scaler.step(optimizer);scaler.update();losses.append(float(loss.detach()))
        pred=predict(va);mae=float(mean_absolute_error(truth[va],pred))
        row=dict(epoch=epoch+1,train_loss=float(np.mean(losses)),validation_mae=mae)
        history.append(row);print('EPOCH '+json.dumps(row),flush=True)
        if mae<best:best=mae;state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};bad=0
        else:bad+=1
        if (bad>=12 and epoch>=20) or time.time()-tick>cfg['budget_seconds']:break
    model.load_state_dict(state);vp=predict(va);tp=predict(np.arange(n,len(programs)))
    cases=pd.DataFrame(dict(row_index=va,program=train.program.iloc[va].tolist(),truth=truth[va],prediction=vp,instructions=lengths[va],absolute_error=np.abs(vp-truth[va])))
    cases.to_csv(OUT/'validation_predictions.csv',index=False)
    groups=cases.groupby(pd.cut(cases.instructions,[0,4,8,16,10000]),observed=True).absolute_error.agg(['mean','count'])
    print('ERROR_BY_LENGTH '+groups.to_json(),flush=True)
    print('VALIDATION_WORST '+cases.nlargest(8,'absolute_error').to_json(orient='records'),flush=True)
    torch.save(dict(state=state,y_mean=ym,y_std=ys,max_args=max_args),OUT/'model.pt')
    sub=pd.DataFrame(dict(ID=test.ID,output=np.rint(tp).astype(np.int64)))
    return finish_run(info,dict(metric='MAE',value=best,direction='minimize',samples=len(va),split='fixed_train_holdout_15pct'),sub,
        dict(ids_unique=sub.ID.is_unique,schema_valid=list(sub.columns)==['ID','output'],row_count=len(sub),finite=bool(np.isfinite(tp).all()),parser_canary=True,no_instruction_truncation=True),history,
        artifacts=dict(tokenizer='train-only BPE',pretrained_weights=False,max_instructions=max_steps,max_operands=max_args,max_tokens_per_instruction=max_tokens,error_by_length=json.loads(groups.to_json()),train_sha256=hash_file(root/'train.csv')),
        limitations=['Architecture inspired by recurrent execution; not a reproduction of NPI, whose trace labels are unavailable.',
                    'Explicit operand values are an arithmetic input bias, not an implemented NAU/NMU.',
                    'R09 changes representation and recurrent direction together; R10 isolates curriculum relative to R09.',
                    'Single fixed holdout and seed; no claim of exact execution or length extrapolation.'])



In [ ]:
OUT = Path('/kaggle/working') / 'lost-interpreter' / f'R{RUN_ID:02d}'
OUT.mkdir(parents=True, exist_ok=True)
report = run_lost(RUN_ID)
print('Submission:', OUT / 'submission.csv')
